<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.3: The Decoder
**Prev: [The ALU, the Immediates and the Registers](5.2_alu_and_regfile.ipynb)**<br>

## Motivation
The decoder turns an instruction into the control signals of the datapath: which ALU operation,
which immediate format, where the ALU's operands come from, and what kind of instruction it is.
It is also where an instruction that is not RV32I is caught.

A decoder can be written by hand, as nested `switch` statements on the opcode and `funct3`, but
then the knowledge "what does `sltiu` do" is spread over many lines. This notebook writes it as a
*table*, one row per instruction, and lets Chisel generate and minimize the logic, with
`chisel3.util.experimental.decode` and the `BitPat`s of [Module 2.7](2.7_enums_and_bundles.ipynb).
At the end, the two styles are compared.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# The control signals

`riscv.Control` holds what the decoder decides; three new `ChiselEnum`s name the choices:

- `kind` (`Kind`): `Alu`, `Load`, `Store`, `Branch`, `Jal`, `Jalr`, `Fence`, `Ecall` or `Ebreak`.
  The datapath derives everything else from it: whether `rd` is written and with what (the ALU
  result, the loaded value, or pc + 4), whether memory is read or written, how the next pc is
  computed. The size of a load or store and the condition of a branch are in `funct3`, so the
  datapath takes them from the instruction directly (a branch compares its registers in a
  comparator of its own: the ALU is busy computing the target);
- `aluOp` (`AluOp`, from 5.2) and `immFormat` (`ImmFormat`, from 5.2);
- `aSel` (`ASel`: `Rs1`, `Pc` or `Zero`) and `bSel` (`BSel`: `Rs2` or `Imm`), the ALU's operands:
  `auipc` and the jump and branch targets add to the pc, `lui` adds to zero;
- `legal`: false for a word that is not an RV32I instruction (the other fields then mean nothing).

---
# The table

`riscv/src/main/scala/riscv/DecodeRows.scala` has one `Row` per instruction. A field that an
instruction does not use is `None`: a *don't care*, which the logic minimizer may set to whatever
makes the logic smallest (`fence`, for example, uses nothing).

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "DecodeRows.scala"))

## Exercise: complete the table

Three rows are missing (`???`): `BGEU`, `SLTIU` and `SRA`. Fill them in in
`riscv/src/main/scala/riscv/DecodeRows.scala`, using the helpers at the top of the table and the
rows of similar instructions. Then **restart the kernel** and run the notebook again. Until then,
the cells that use the table stop with a `NotImplementedError` (the `???`).

The first check compares the table with a second description of the same decoder, worked out
from each instruction's opcode in the encodings of [Module 5.1](5.1_riscv_isa.ipynb): the opcode
gives the kind and the operands, the format gives the immediate, and an arithmetic instruction
asks the ALU for its own operation (`sltiu` for `Sltu`). A wrong or missing field fails it.

In [ ]:
// the row an instruction must have, worked out from its opcode and name alone
def expectedRow(name: String): Row = {
  val e   = Rv32i.encodings(name.toLowerCase)
  val alu = AluOp.all.find(AluOp.mnemonic(_) == name.toLowerCase.filterNot(_ == 'i')) // addi -> add
  def row(kind: Kind.Type, a: ASel.Type, b: BSel.Type, op: Option[AluOp.Type] = Some(AluOp.Add)) =
    Row(kind, op, ImmFormat.of(e.format), Some(a), Some(b))
  e.opcode match {
    case 0x37 => row(Kind.Alu, ASel.Zero, BSel.Imm) // lui: 0 + imm
    case 0x17 => row(Kind.Alu, ASel.Pc, BSel.Imm)   // auipc: pc + imm
    case 0x6f => row(Kind.Jal, ASel.Pc, BSel.Imm)   // the target
    case 0x67 => row(Kind.Jalr, ASel.Rs1, BSel.Imm)
    case 0x63 => row(Kind.Branch, ASel.Pc, BSel.Imm)
    case 0x03 => row(Kind.Load, ASel.Rs1, BSel.Imm) // the address
    case 0x23 => row(Kind.Store, ASel.Rs1, BSel.Imm)
    case 0x13 => row(Kind.Alu, ASel.Rs1, BSel.Imm, alu)
    case 0x33 => row(Kind.Alu, ASel.Rs1, BSel.Rs2, alu)
    case 0x0f => Row(Kind.Fence)
    case 0x73 => Row(if (name == "ECALL") Kind.Ecall else Kind.Ebreak)
  }
}
// rows as plain values: enum values compare by their encoding
def values(r: Row) =
  (r.kind.litValue, r.aluOp.map(_.litValue), r.imm.map(_.litValue), r.a.map(_.litValue), r.b.map(_.litValue))

val rows = DecodeRows.table
assert(rows.keySet == Instructions.all.map(_._1).toSet, "a row for every instruction")
for ((name, row) <- rows) assert(values(row) == values(expectedRow(name)), s"the row of $name")
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
    "BGEU"  -&gt; branch,
    "SLTIU" -&gt; imm(Sltu),
    "SRA"    -&gt; reg(Sra),
</pre></article></div></section></div>

---
# From the table to logic

`chisel3.util.experimental.decode` turns the table into hardware in three steps (see the
`Decoder` object in `riscv/src/main/scala/riscv/Decoder.scala`):

1. each instruction is a `DecodePattern`, here `InstPattern(name, bitPat)` with the `BitPat` of
   `Instructions.all`;
2. each control signal is a `DecodeField`, whose `genTable(pattern)` gives the signal's value for
   that instruction as a `BitPat`: `BitPat(AluOp.Add)` for a value, `dc` (don't care) for `None`.
   The `legal` field is `y` for every instruction, and its `default` (for a word that matches no
   pattern) is `n`;
3. `DecodeTable(patterns, fields)` concatenates the fields into one output word per pattern: a
   *truth table*, with the defaults for every other input.

A logic minimizer then turns the truth table into a sum of products. `decoder.qmc` uses
Quine-McCluskey, built into Chisel; `DecodeTable.decode` would first try the external `espresso`
program, which is not installed here (it would fall back to Quine-McCluskey with a warning), so
the module calls `decoder.qmc` directly.

The truth table has one line per instruction, the input pattern then the output bits:

In [ ]:
val truth = Decoder.table.table
println(s"${truth.table.size} rows, ${truth.table.head._1.getWidth} input bits, ${truth.default.getWidth} output bits")
truth.table.take(3).foreach { case (in, out) => println(s"$in -> $out") }

---
# Testing the decoder

Two tests in one simulation: 50 random instances of every RV32I instruction must give the
signals of their row (where the row cares), and words that are not RV32I must be flagged as
illegal. The illegal words include *near misses*, instructions with one fixed bit flipped, which
random words would hardly ever produce; the software decoder of 5.1 says which words are not RV32I.

In [ ]:
simulate(new Decoder) { c =>
  val rows = DecodeRows.table // each cell stands alone: an earlier one may have failed
  val rnd  = new scala.util.Random(1)
  for ((name, _) <- Instructions.all; _ <- 0 until 50) {
    val word = Instructions.random(name, rnd)
    val row  = rows(name)
    val what = f"$name 0x$word%08x"
    c.io.inst.poke(word.U)
    c.io.ctrl.legal.expect(true.B, what)
    c.io.ctrl.kind.expect(row.kind, what)
    row.aluOp.foreach(c.io.ctrl.aluOp.expect(_, what))
    row.imm.foreach(c.io.ctrl.immFormat.expect(_, what))
    row.a.foreach(c.io.ctrl.aSel.expect(_, what))
    row.b.foreach(c.io.ctrl.bSel.expect(_, what))
  }
  val nearMisses = Instructions.all.flatMap { case (n, _) => Seq.fill(20)(Instructions.nearMiss(n, rnd)) }
  val illegal    = (nearMisses ++ Seq.fill(2000)(rnd.nextLong() & 0xffffffffL)).filter(Rv32i.decode(_).isEmpty)
  for (word <- illegal) {
    c.io.inst.poke(word.U)
    c.io.ctrl.legal.expect(false.B, f"0x$word%08x")
  }
  println(s"${Instructions.all.size * 50} instructions and ${illegal.size} illegal words checked")
}
println("SUCCESS!!")

A table is only correct if no word matches two rows. The patterns of `Instructions.all` must not
overlap: `BitPat.overlap` tells whether some word matches both (this does not need the table, so
it runs before the exercise is done too).

In [ ]:
val overlaps = for {
  (a, i) <- Instructions.all.zipWithIndex
  b      <- Instructions.all.drop(i + 1)
  if a._2.overlap(b._2)
} yield s"${a._1} and ${b._1}"
assert(overlaps.isEmpty, overlaps.mkString(", "))
println(s"no two of the ${Instructions.all.size} patterns overlap")

---
# Table or `switch`?

Here is the same decoder written by hand: a `switch` on the opcode, with `funct3` and `funct7`
inside. It is shorter to type than the table, but each instruction's behavior is spread over
several lines, and the legality conditions (which `funct3` values exist for loads, which `funct7`
for shifts) are easy to get wrong.

In [ ]:
class SwitchDecoder extends Module {
  val io = IO(new Bundle {
    val inst = Input(UInt(32.W))
    val ctrl = Output(new Control)
  })
  val inst             = io.inst
  val (opcode, f3, f7) = (inst(6, 0), inst(14, 12), inst(31, 25))
  val c                = io.ctrl
  c.legal     := false.B
  c.kind      := Kind.Alu
  c.aluOp     := AluOp.Add
  c.immFormat := ImmFormat.I
  c.aSel      := ASel.Rs1
  c.bSel      := BSel.Imm
  // the ALU operation that funct3 selects (funct7 bit 5 turns add into sub and srl into sra)
  val byF3 = VecInit(AluOp.Add, AluOp.Sll, AluOp.Slt, AluOp.Sltu, AluOp.Xor, AluOp.Srl, AluOp.Or, AluOp.And)(f3)
  switch(opcode) {
    is("b0110111".U) { c.legal := true.B; c.immFormat := ImmFormat.U; c.aSel := ASel.Zero } // lui
    is("b0010111".U) { c.legal := true.B; c.immFormat := ImmFormat.U; c.aSel := ASel.Pc } // auipc
    is("b1101111".U) { c.legal := true.B; c.kind := Kind.Jal; c.immFormat := ImmFormat.J; c.aSel := ASel.Pc }
    is("b1100111".U) { c.legal := f3 === 0.U; c.kind := Kind.Jalr }
    is("b1100011".U) {
      c.legal     := f3 =/= 2.U && f3 =/= 3.U
      c.kind      := Kind.Branch
      c.immFormat := ImmFormat.B
      c.aSel      := ASel.Pc
    }
    is("b0000011".U) { c.legal := f3 =/= 3.U && f3 < 6.U; c.kind := Kind.Load }
    is("b0100011".U) { c.legal := f3 < 3.U; c.kind := Kind.Store; c.immFormat := ImmFormat.S }
    is("b0010011".U) { // addi ... srai
      val shift = f3 === 1.U || f3 === 5.U
      c.legal := !shift || f7 === 0.U || (f3 === 5.U && f7 === "b0100000".U)
      c.aluOp := Mux(f3 === 5.U && f7(5), AluOp.Sra, byF3)
    }
    is("b0110011".U) { // add ... and
      c.legal := f7 === 0.U || (f7 === "b0100000".U && (f3 === 0.U || f3 === 5.U))
      c.aluOp := Mux(f7(5), Mux(f3 === 0.U, AluOp.Sub, AluOp.Sra), byF3)
      c.bSel  := BSel.Rs2
    }
    is("b0001111".U) { c.legal := f3 === 0.U; c.kind := Kind.Fence }
    is("b1110011".U) {
      c.legal := inst === "h00000073".U || inst === "h00100073".U
      c.kind  := Mux(inst(20), Kind.Ebreak, Kind.Ecall)
    }
  }
}

The two decoders must agree wherever the table cares, and on legality. Their size, measured
roughly as lines of FIRRTL and of SystemVerilog after `firtool`'s optimizations (a fairer
measure, the number of logic cells after synthesis, needs Yosys, which comes in Module 5.14):

In [ ]:
// both decoders on the same instruction; one output per signal says whether they agree
class Compare extends Module {
  val inst = IO(Input(UInt(32.W)))
  val same = IO(Output(new Bundle {
    val legal, kind, aluOp, immFormat, aSel, bSel = Bool()
  }))
  val table = Module(new Decoder)
  val hand  = Module(new SwitchDecoder)
  table.io.inst := inst
  hand.io.inst  := inst
  val (t, h) = (table.io.ctrl, hand.io.ctrl)
  same.legal     := t.legal === h.legal
  same.kind      := t.kind === h.kind
  same.aluOp     := t.aluOp === h.aluOp
  same.immFormat := t.immFormat === h.immFormat
  same.aSel      := t.aSel === h.aSel
  same.bSel      := t.bSel === h.bSel
}
simulate(new Compare) { c =>
  val rows  = DecodeRows.table
  val rnd   = new scala.util.Random(3)
  val words = Instructions.all.flatMap { case (n, _) =>
    Seq.fill(20)(Instructions.random(n, rnd)) ++ Seq.fill(20)(Instructions.nearMiss(n, rnd))
  } ++ Seq.fill(2000)(rnd.nextLong() & 0xffffffffL)
  for (word <- words) {
    c.inst.poke(word.U)
    val what = f"0x$word%08x"
    c.same.legal.expect(true.B, what)
    for (i <- Rv32i.decode(word); row = rows(i.name.toUpperCase)) { // where the row cares
      c.same.kind.expect(true.B, what)
      if (row.aluOp.isDefined) c.same.aluOp.expect(true.B, what)
      if (row.imm.isDefined) c.same.immFormat.expect(true.B, what)
      if (row.a.isDefined) c.same.aSel.expect(true.B, what)
      if (row.b.isDefined) c.same.bSel.expect(true.B, what)
    }
  }
}
def size(gen: => RawModule): (Int, Int) = (
  _root_.circt.stage.ChiselStage.emitCHIRRTL(gen).linesIterator.size,
  _root_.circt.stage.ChiselStage
    .emitSystemVerilog(gen, firtoolOpts = Array("-disable-all-randomization", "-strip-debug-info"))
    .linesIterator
    .size
)
val (tableFir, tableSv) = size(new Decoder)
val (handFir, handSv)   = size(new SwitchDecoder)
println(s"table-driven: $tableFir lines of FIRRTL, $tableSv lines of SystemVerilog")
println(s"switch:       $handFir lines of FIRRTL, $handSv lines of SystemVerilog")
println("SUCCESS!!")

The `switch` comes out several times smaller as text. The minimizer turns each of the 15 output
bits into a sum of products of the instruction bits on its own, using every don't care; products
that come out identical are shared, but there is nothing like the `switch`'s structure, where one
comparison of the opcode serves all the signals of an instruction class. Lines of text are not gates, though: a synthesis tool optimizes both
descriptions again, and whether the gate counts differ as much is for Yosys to say (Module 5.14).

The table's advantage is elsewhere: adding an instruction is adding a row, and the tests above
check every row against an independent description of the ISA. The processors of this course
use the table-driven `Decoder`.

---
# You're done!

The next notebook puts the ALU, the immediate generator, the register file and the decoder
together into a single-cycle RV32I processor.

[Return to the top.](#top)